# Steve's Luxury Resort — Churn Prediction

**MMAI 869 Team Project · Team-Union**

This notebook walks through the team's analysis: data exploration, feature engineering, modeling, and final ensemble.

**Competition:** [Steve's Luxury Resort (MMAI)](https://www.kaggle.com/competitions/steves-luxury-resort-mmai) on Kaggle.
**Metric:** F-Score (Macro)
**Deliverable:** 12-minute live presentation.

## 1. Setup

In [14]:
import sys
sys.path.insert(0, '..')

import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import urllib.request
from pathlib import Path


#from src.data import load_raw, basic_clean, split_xy, ID_COL, TARGET
#from src.features import add_all_features

def load_churn_data():
    # Load the data from the local file, or download it if it doesn't exist
    churn_path = Path("/Users/nick/Library/CloudStorage/OneDrive-Personal/Programming projects/Team Union/datasets/resort_test.csv")
    if not churn_path.is_file():
        Path("datasets").mkdir(parents=True, exist_ok=True)
        url = "https://github.com/stepthom/869_course/blob/main/data/resort_test.csv"
        urllib.request.urlretrieve(url, churn_path)
    return pd.read_csv(churn_path)

churn = load_churn_data()

sns.set_theme(style='whitegrid', context='talk')
pd.set_option('display.max_columns', 80)

In [15]:
#code to save the figures as high-res PNGs

IMAGES_PATH = Path() / "images" / "test_data_clean_and_validation"
IMAGES_PATH.mkdir(parents=True, exist_ok=True)

def save_fig(fig_id, tight_layout=True, fig_extension="png", resolution=300):
    path = IMAGES_PATH / f"{fig_id}.{fig_extension}"
    if tight_layout:
        plt.tight_layout()
    plt.savefig(path, format=fig_extension, dpi=resolution)

In [16]:
#the next 5 lines define the default font sizes of graphs
plt.rc('font', size=14)
plt.rc('axes', labelsize=14, titlesize=14)
plt.rc('legend', fontsize=14)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)

## 2. Load and inspect

In [17]:
#train_raw, test_raw, sample = load_raw()
#print(f'train: {train_raw.shape}, test: {test_raw.shape}, submission: {sample.shape}')
churn.head()

,GuestID,BookingDate,PromoCode,Region,AllInclusive,Room,PackageType,Age,VIP,RoomService,Dining,Retail,Spa,Entertainment,LoyaltyPoints,SurveyScore,DaysSinceEmail,BookingChannel,AgeGroup,ReferralSource
0,154038,2024-12-12,NaN,Europe,1.0,E/230/P,Adventure,34.0,0.0,0.0,0.0,NaN,0.0,0.000000,8473,2,65,TravelAgent,Middle,Facebook
1,620160,2024-05-26,PromoA,Americas,1.0,G/1242/S,Relaxation,4.0,0.0,NaN,0.0,0.0,0.0,0.000000,5312,4,319,TravelAgent,Minor,Friend
2,655103,2023-07-20,NaN,AsiaPacific,0.0,F/1766/S,Relaxation,25.0,0.0,410.0,32.0,14.0,1239.0,10.000000,1746,5,186,Mobile,Young,Yelp
3,126993,2024-03-15,PromoB,AsiaPacific,0.0,F/1319/S,Relaxation,12.0,NaN,NaN,0.0,0.0,0.0,234148.872408,5555,4,129,TravelAgent,Minor,Instagram
4,635228,2024-12-23,PromoA,Europe,0.0,E/556/S,Adventure,66.0,1.0,0.0,1828.0,1.0,1873.0,45.000000,3631,5,264,TravelAgent,Elderly,Podcast


In [18]:
churn.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1739 entries, 0 to 1738
Data columns (total 20 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   GuestID         1739 non-null   int64  
 1   BookingDate     1739 non-null   object 
 2   PromoCode       951 non-null    object 
 3   Region          1707 non-null   object 
 4   AllInclusive    1690 non-null   float64
 5   Room            1651 non-null   object 
 6   PackageType     1710 non-null   object 
 7   Age             1612 non-null   float64
 8   VIP             1694 non-null   float64
 9   RoomService     1627 non-null   float64
 10  Dining          1614 non-null   float64
 11  Retail          1700 non-null   float64
 12  Spa             1707 non-null   float64
 13  Entertainment   1695 non-null   float64
 14  LoyaltyPoints   1739 non-null   int64  
 15  SurveyScore     1739 non-null   int64  
 16  DaysSinceEmail  1739 non-null   int64  
 17  BookingChannel  1739 non-null   o

## 3. Begin by filling in as much information in the age and AgeGroup

We can use information contained in other columns to extrapolate some of the null information

**Information from the AgeGroup to fill in the Age column will skew the data slightly, be careful if you want to make conclusions based off the age column**

In [19]:
churn["AgeGroup"].value_counts()

AgeGroup
Young      652
Middle     450
Minor      315
Senior     204
Elderly     44
Name: count, dtype: int64

In [20]:
#count the number of null values in AgeGroup column and  Age column
print(churn["AgeGroup"].isnull().sum())
print(churn["Age"].isnull().sum())

74
127


### Data Cleaning - Missing values - Age Group and Age

Minor      1372 1-18    10

Young      2586 19-30   24

Middle     1810 31-45   38

Senior      727 46-60   52

Elderly     176 61-79   70

we can use this data to fill in blanks in either group - I'll use the median age for the groups to fill in missing age values

In [21]:
#find null values in AgeGroup column and for each corresponding row, fill in the agegroup based on the age column. "Minor" if age >0 && <= 18, "Young" if age > 18 && <=30, "Middle" if age > 30 && <= 45, "Senior" if age > 45 && <= 60, "Elderly" if age > 60. 

mask = churn['AgeGroup'].isna() & churn['Age'].notna()

churn.loc[mask, 'AgeGroup'] = pd.cut(
    churn.loc[mask, 'Age'],
    bins=[0, 18, 30, 45, 60, float('inf')],
    labels=['Minor', 'Young', 'Middle', 'Senior', 'Elderly'],
    right=True,            # right=True means upper bound is inclusive: (lo, hi]
    include_lowest=True,  # age must be >= 0
)

print(churn["AgeGroup"].isnull().sum())
print(churn["Age"].isnull().sum())

41
127


In [22]:
#now we need to do the same thing for the Age column, find null values in Age column and for each corresponding row, fill in the age based on the agegroup column. 10 if agegroup is "Minor", 24 if agegroup is "Young", 38 if agegroup is "Middle", 52 if agegroup is "Senior", 70 if agegroup is "Elderly".
mask = churn['Age'].isna() & churn['AgeGroup'].notna()
churn.loc[mask, 'Age'] = churn.loc[mask, 'AgeGroup'].map({
    'Minor': 10,
    'Young': 24,
    'Middle': 38,
    'Senior': 52,
    'Elderly': 70
})

print(churn["AgeGroup"].isnull().sum())
print(churn["Age"].isnull().sum())

#The displayed 2 numbers should both be the same, this means that the remaining values are null in both columns and we cannot fill them in based on the other column. We may have to drop these rows later on.

41
41


## 4. Is there any information we can get from guests that shared a room?

In [23]:
#I want to know how many guests shared a room
#Create a new column called "SharedRoom" that is contains a 1 if there is another GuesID that shares the same Room and BookingDate, and 0 otherwise.
churn['SharedRoom'] = churn.duplicated(subset=['Room', 'BookingDate'], keep=False).astype(int)

#Count the number of guests that shared a room
num_shared = churn['SharedRoom'].sum()
print(f'Number of guests that shared a room: {num_shared}')

Number of guests that shared a room: 21


## 5. Do promo codes affect churn rates?

In [24]:
churn['PromoCodeUsed'] = churn['PromoCode'].map({'PromoA': 1, 'PromoB': 2}).fillna(0).astype(int)

## 11. Are responsive people more likely to churn?

In [25]:
#generate a figure that compares DaysSinceEmail to Churned. Use a deciles to show the distribution of DaysSinceEmail for both churned and non-churned customers.
churn['DaysSinceEmail_Decile'] = pd.qcut(churn['DaysSinceEmail'], q=10, labels=False)

# We have skewed the age column so let's drop it

In [26]:
#drop the Age column
churn = churn.drop("Age", axis=1)

## Export the test

In [27]:
#Export the churned data to a csv file called "churn_test_cleaned.csv"
churn.to_csv("churn_test_cleaned.csv", index=False)


